# Simulación Piloto JTM — Dataset Sintético de Check-in (v3)

**Propósito.** Este notebook documenta, de forma reproducible, la generación del dataset sintético
`jtm_checkin_sintetico_v3.csv`, usado para que el equipo de ingeniería de Pymtech pueda construir y
probar el motor analítico de ARQIA (ingestión, cálculo de ISP, reglas de disparo) **antes** de tener
acceso a datos reales de colaboradores de JTM.

**Advertencia de origen — léase antes de usar este archivo.**
Este dataset es **100% simulado**, generado en esta sesión de trabajo con semilla fija (`seed=42`)
para que sea reproducible. No proviene de ningún repositorio externo ni de un flujo de Google Colab
previo: se generó aquí mismo, el 25 de noviembre de 2026, siguiendo exactamente la especificación
de ARQIA (42 colaboradores, 3 departamentos, 8 semanas, 5 dimensiones). El objetivo es que Pymtech
tenga un artefacto real y verificable sobre el cual construir, no una referencia a algo que no se
puede inspeccionar.

**La 5ª dimensión (`profissional`).** El check-in real de JTM, tal como está documentado en el
framing oficial compartido por el CEO, tiene 4 dimensiones (Personal, Salud, Energía, Desafío de
la semana). La inclusión de una 5ª dimensión (`profissional`, antes de `desafio`) responde a una
solicitud informal del CEO de JTM por WhatsApp (25 nov 2026), **todavía sin confirmación escrita**.
Este dataset ya incluye esa 5ª dimensión para que el equipo de ingeniería pueda diseñar el esquema
con ella desde ahora, pero el SRS que acompaña este dataset dejará explícita esta trazabilidad y
recomendará obtener la confirmación por escrito antes de que esto se congele como especificación
definitiva.


In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

DEPTS = [
    ("Operações Portuárias", 15, "JTM_{:03d}"),
    ("Gestão de Frota", 15, "JTM_{:03d}"),
    ("Direção e Comercial", 12, "JTM_{:03d}"),
]
WEEKS = [f"W{str(i).zfill(2)}" for i in range(1, 9)]
DIMS = ["pessoal", "saude", "energia", "profissional", "desafio"]
WEIGHTS = {"pessoal": 0.15, "saude": 0.25, "energia": 0.25, "profissional": 0.25, "desafio": 0.10}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9

CHALLENGE_TEXTS = [
    "Consegui entregar o relatório a tempo, mas com muita pressão.",
    "Semana tranquila, sem grandes desafios de destaque.",
    "Tive dificuldade em coordenar a equipa no turno da tarde.",
    "Resolvi um conflito com um colega, correu melhor do que esperava.",
    "Senti-me sobrecarregado com o volume de trabalho esta semana.",
    "Consegui equilibrar bem o trabalho e o descanso.",
    "Faltou apoio da liderança numa decisão importante.",
    "Boa semana, sinto que avancei no meu desenvolvimento.",
    "Muita pressão com os prazos de entrega do porto.",
    "Tive uma conversa difícil com o meu responsável direto.",
    "Consegui finalizar a formação do módulo sem atrasos.",
    "Senti-me desmotivado, sem energia para os desafios habituais.",
]

def clip(v):
    return int(np.clip(round(v), 1, 9))


## Patrones simulados (anomalías de diseño)

Para que el motor analítico tenga algo real que detectar en la demo, el dataset incluye **tres
patrones deliberados**, cada uno pensado como ilustración de una regla de negocio distinta:

1. **Desgaste estructural** — 5 colaboradores de *Operações Portuárias* muestran un declive
   sostenido en `saude`, `energia` y `profissional` durante `W06`–`W08`. Ilustra la regla de
   "tendencia negativa ≥ 2 semanas consecutivas".
2. **Alerta crítica** — 2 colaboradores de *Gestão de Frota* puntúan ≤ 3 en casi todas las
   dimensiones durante `W03`–`W05` (3 semanas consecutivas). Ilustra la regla de disparo de
   oferta de apoyo privado (nunca visible para el manager directo).
3. **Sesgo de respuesta plana** — 4 colaboradores de *Direção e Comercial* responden siempre `8`
   en las 5 dimensiones, las 8 semanas. Ilustra la regla de "varianza cero → posible desconexión
   o respuesta de compromiso, no de reflexión real", que el motor debe marcar para revisión
   cualitativa en vez de tratarla como una señal positiva genuina.

El resto de la población (31 de 42 colaboradores) tiene una línea base individual (~7, con ruido
gaussiano moderado) sin patrón forzado, para que el dataset no sea *solo* casos extremos.


In [2]:
records = []
uid_counter = 1

for dept, n, fmt in DEPTS:
    dept_ids = [fmt.format(uid_counter + i) for i in range(n)]
    uid_counter += n

    if dept == "Operações Portuárias":
        wear_subset = set(dept_ids[:5])
        alert_subset, flat_subset = set(), set()
    elif dept == "Gestão de Frota":
        alert_subset = set(dept_ids[:2])
        wear_subset, flat_subset = set(), set()
    else:
        flat_subset = set(dept_ids[:4])
        wear_subset, alert_subset = set(), set()

    baselines = {uid: 7 + rng.normal(0, 0.6) for uid in dept_ids}

    for uid in dept_ids:
        base = baselines[uid]
        for wi, wk in enumerate(WEEKS, start=1):
            if uid in flat_subset:
                vals = {d: 8 for d in DIMS}
            elif uid in wear_subset and wk in ("W06", "W07", "W08"):
                decline = {"W06": 1.2, "W07": 2.2, "W08": 3.0}[wk]
                vals = {
                    "pessoal": clip(base + rng.normal(0, 0.5)),
                    "saude": clip(base - decline + rng.normal(0, 0.4)),
                    "energia": clip(base - decline + rng.normal(0, 0.4)),
                    "profissional": clip(base - decline + rng.normal(0, 0.4)),
                    "desafio": clip(base - decline * 0.6 + rng.normal(0, 0.5)),
                }
            elif uid in alert_subset and wk in ("W03", "W04", "W05"):
                vals = {d: clip(rng.uniform(1.5, 3.0)) for d in DIMS}
            else:
                vals = {d: clip(base + rng.normal(0, 0.9)) for d in DIMS}

            weighted = sum(vals[d] * WEIGHTS[d] for d in DIMS)
            isp_score = round((weighted / 9.0) * 100, 1)

            texto = rng.choice(CHALLENGE_TEXTS)
            records.append({
                "user_hash": uid, "departamento": dept, "semana": wk,
                "pessoal": vals["pessoal"], "saude": vals["saude"], "energia": vals["energia"],
                "profissional": vals["profissional"], "desafio": vals["desafio"],
                "isp_score": isp_score, "desafio_texto": texto,
            })

df = pd.DataFrame(records)
assert len(df) == 336
df.to_csv("jtm_checkin_sintetico_v3.csv", index=False)
print(f"Generado: {len(df)} registros, {df.user_hash.nunique()} colaboradores, {df.semana.nunique()} semanas.")
df.head()


Generado: 336 registros, 42 colaboradores, 8 semanas.


,user_hash,departamento,semana,pessoal,saude,energia,profissional,desafio,isp_score,desafio_texto
0,JTM_001,Operações Portuárias,W01,6,8,6,8,7,78.9,"Semana tranquila, sem grandes desafios de dest..."
1,JTM_001,Operações Portuárias,W02,7,8,7,7,7,80.6,Tive uma conversa difícil com o meu responsáve...
2,JTM_001,Operações Portuárias,W03,8,8,8,8,9,90.0,Senti-me sobrecarregado com o volume de trabal...
3,JTM_001,Operações Portuárias,W04,7,6,8,8,7,80.6,"Senti-me desmotivado, sem energia para os desa..."
4,JTM_001,Operações Portuárias,W05,6,6,8,8,8,80.0,Tive uma conversa difícil com o meu responsáve...


## Validación 1 — Desgaste estructural (Operações Portuárias)

In [3]:
wear_ids = [f"JTM_{i:03d}" for i in range(1, 6)]
sub = df[(df.departamento == "Operações Portuárias") & (df.user_hash.isin(wear_ids))]
sub.groupby("semana")[["saude", "energia", "profissional"]].mean().round(2)


,saude,energia,profissional
semana,,,
W01,6.8,6.8,6.8
W02,7.2,7.2,7.0
W03,6.4,6.6,7.0
W04,6.8,7.6,7.2
W05,6.8,7.0,6.6
W06,6.0,5.6,5.6
W07,4.8,5.0,5.0
W08,3.8,3.6,4.2


## Validación 2 — Alerta crítica (Gestão de Frota, W03–W05 ≤ 3)

In [4]:
alert_ids = [f"JTM_{i:03d}" for i in range(16, 18)]
sub2 = df[(df.departamento == "Gestão de Frota") & (df.user_hash.isin(alert_ids)) & (df.semana.isin(["W03", "W04", "W05"]))]
print("Máximo valor en la ventana crítica:", sub2[["pessoal","saude","energia","profissional","desafio"]].max().max())
sub2[["user_hash", "semana", "pessoal", "saude", "energia", "profissional", "desafio"]]


Máximo valor en la ventana crítica: 3


,user_hash,semana,pessoal,saude,energia,profissional,desafio
122,JTM_016,W03,2,2,2,2,2
123,JTM_016,W04,2,3,2,2,2
124,JTM_016,W05,2,2,2,3,2
130,JTM_017,W03,2,3,2,3,2
131,JTM_017,W04,3,3,2,3,2
132,JTM_017,W05,2,3,2,3,2


## Validación 3 — Sesgo de respuesta plana (Direção e Comercial)

In [5]:
flat_ids = [f"JTM_{i:03d}" for i in range(31, 35)]
sub3 = df[(df.departamento == "Direção e Comercial") & (df.user_hash.isin(flat_ids))]
print("Conjunto de valores únicos (dims x semanas):", set(sub3[["pessoal","saude","energia","profissional","desafio"]].values.flatten()))
print("isp_score único:", sub3["isp_score"].unique())


Conjunto de valores únicos (dims x semanas): {np.int64(8)}
isp_score único: [88.9]


## Resumen por departamento

Distribución final de colaboradores y promedio de ISP por departamento, como chequeo de forma
antes de entregar el archivo al equipo de ingeniería.


In [6]:
resumen = df.groupby("departamento").agg(
    colaboradores=("user_hash", "nunique"),
    registros=("user_hash", "size"),
    isp_promedio=("isp_score", "mean"),
).round(1)
resumen


,colaboradores,registros,isp_promedio
departamento,,,
Direção e Comercial,12,96,81.2
Gestão de Frota,15,120,76.9
Operações Portuárias,15,120,75.1


## Nota final

Este archivo y este notebook son el artefacto real que Pymtech debe usar para construir el pipeline
de ingestión y el motor analítico del piloto. Cuando JTM entregue datos reales de colaboradores
(bajo el contrato de ingesta descrito en el SRS v2.1), este mismo esquema de columnas debe aceptarlos
sin cambios estructurales — esa es precisamente la razón de ser del "Diccionario de Datos y Contrato
de Ingesta" del SRS.
